In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
SHA2_BITS = 256

In [0]:
%run ./../common/utilities

In [0]:
dbutils.widgets.text("catalog", "abcgroup", "Catalog")

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
crm_products = f"{catalog}.{silver_schema}.crm_products"
erp_product_cat = f"{catalog}.{silver_schema}.erp_product_category"

In [0]:
df = (
    spark.table(crm_products).alias("pn")
    .join(
        spark.table(erp_product_cat).alias("pc"),
        F.col("pn.category_id") == F.col("pc.category_id"),
        "left"
    )
    .select(
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("pn.product_id"), F.lit("")),
                F.coalesce(F.col("pn.product_number"), F.lit("")),
                F.coalesce(F.col("pn.start_date").cast("string"), F.lit(""))
            ),
            SHA2_BITS
        ).alias("product_key"),

        F.col("pn.product_id"),
        F.col("pn.product_number"),
        F.col("pn.product_name"),
        F.col("pn.category_id"),
        F.col("pc.category"),
        F.col("pc.subcategory"),
        F.col("pc.maintenance_flag"),
        F.col("pn.product_line"),
        F.col("pn.start_date")
    )
)

In [0]:
display(df.limit(5))

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{gold_schema}.dim_products")

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {catalog}.{gold_schema}.dim_products
    LIMIT 5
"""))